# W11-D2 概念实验：Gap Matrix

核心概念来自 Markdown：Gap Matrix 是风险地图，不是完成度报表。Supply Chain 最薄弱，KnowledgeSnapshot、CapabilityRelease、RuntimeABI 等对象会影响部署闭包的可复现性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
layers = ["Business Domain", "Supply Chain", "Runtime", "Operations"]
objects = ["Capability", "KnowledgeSnapshot", "CapabilityRelease", "RuntimeABI", "DeploymentRevision", "Connector"]
# 评分遵循文档：9-10 生产级，5-6 骨架级，1-2 空白
scores = np.array([
    [7, 2, 2, 2, 7, 0],
    [6, 2, 2, 2, 5, 2],
    [4, 2, 2, 2, 7, 0],
    [6, 2, 2, 2, 5, 2],
], dtype=float)
print("各层平均分:", {layer: round(float(row[row > 0].mean()), 2) for layer, row in zip(layers, scores)})
print("全局最高风险对象:", objects[int(np.argmin(np.where(scores == 0, 99, scores).min(axis=0)))])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
masked = np.ma.masked_equal(scores, 0)
im = ax.imshow(masked, cmap="RdYlGn", vmin=1, vmax=10, aspect="auto")
ax.set_xticks(range(len(objects)), objects, rotation=30, ha="right")
ax.set_yticks(range(len(layers)), layers)
ax.set_title("目标态对象实现度 Gap Matrix（模拟评分）")
for i in range(scores.shape[0]):
    for j in range(scores.shape[1]):
        label = "空白" if scores[i, j] == 0 else f"{scores[i, j]:.0f}"
        ax.text(j, i, label, ha="center", va="center", fontsize=10)
fig.colorbar(im, ax=ax, label="实现度评分（1-10）")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# 用“低分 × 架构影响”把完成度转换为风险优先级
impact = np.array([2, 5, 4, 5, 5, 4], dtype=float)
min_scores = np.where(scores == 0, 1, scores).min(axis=0)
risk = impact * (11 - min_scores)
priority = sorted(zip(risk, objects, min_scores), reverse=True)
for value, obj, score in priority:
    print(f"{obj:18s} score={score:.0f} risk={value:.0f}")
print("结论：先补 KnowledgeSnapshot/RuntimeABI 等会破坏闭包或兼容性的语义 Gap，而不是按模块行数排期。")